# Exp 11: ResNet-18 5-fold

Card: [exp11_resnet18_5fold.md](../../experiments/exp11_resnet18_5fold.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, CVResult, Config, CrossValidator
from core.scores import Scores
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-resnet18-5fold-smoke",
        "folds": [0, 1],
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
else:
    OVERRIDES = {}
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.toml", **OVERRIDES
)
EXP09 = Config.from_toml(CONFIG_DIR / "exp09_full_ratio_weight" / "resnet18.toml")
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp11_resnet18_5fold"
SAVE_FIGURES = not SMOKE
PROBS = [f"p_{c}" for c in CLASSES]

# dataviz reference palette, light mode; ResNet-18 keeps its color from exp10
COLOR = "#2a78d6"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training


In [ ]:
result = CrossValidator(CONFIG).run()
oof = result.oof

## Per-fold scores


In [ ]:
def fold_scores(result):
    """AP per class, mAP and Macro F1 (best-F1 thresholds tuned on same fold rows), one row per fold."""
    macro_f1 = {
        fold: Scores(part[CLASSES].to_numpy(), part[PROBS].to_numpy()).best_f1()["f1"].mean()
        for fold, part in result.oof.groupby("fold")
    }
    return result.fold_ap.assign(**{"Macro F1": pd.Series(macro_f1)})


scores = fold_scores(result)
summary = pd.concat([scores, scores.agg(["mean", "std"])])
summary.round(3)

## Run-to-run noise


In [ ]:
# same setup, same folds, trained twice
noise = (scores - fold_scores(CVResult.load(EXP09))).dropna().T.add_prefix("fold ")
noise.round(3)

## Macro F1 on all out-of-fold rows


In [ ]:
same = result.scores.best_f1()

# each fold scored with thresholds tuned on rest
pred = pd.DataFrame(0, index=oof.index, columns=CLASSES)
for fold in CONFIG.folds:
    rest, part = oof[oof["fold"] != fold], oof["fold"] == fold
    thresholds = Scores(rest[CLASSES].to_numpy(), rest[PROBS].to_numpy()).best_f1()["threshold"]
    pred.loc[part] = (oof.loc[part, PROBS].to_numpy() >= thresholds.to_numpy()).astype(
        int
    )
cross = pd.Series(
    f1_score(oof[CLASSES], pred, average=None, zero_division=0),
    index=CLASSES,
    name="f1",
)

pooled = pd.concat(
    {
        "tuned on same rows": same[["threshold", "f1"]],
        "tuned on other folds": cross.to_frame(),
    },
    axis=1,
)
pooled.loc["Macro F1"] = pooled.mean()
pooled.loc["Macro F1", ("tuned on same rows", "threshold")] = float("nan")
pooled.round(3)

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(1, len(VIP), figsize=(3.6 * len(VIP), 3.6), sharey=True)
for cls, ax in zip(VIP, axes):
    precision, recall, _ = precision_recall_curve(oof[cls], oof[f"p_{cls}"])
    ax.plot(recall, precision, color=COLOR, lw=2, drawstyle="steps-post")
    best = same.loc[cls]
    ax.plot(
        best["recall"], best["precision"], "o", ms=8, color=COLOR, mec="#fcfcfb", mew=2
    )
    ax.annotate(
        f"F1 {best['f1']:.3f}\nt {best['threshold']:.3f}",
        (best["recall"], best["precision"]),
        xytext=(-8, -10),
        textcoords="offset points",
        ha="right",
        va="top",
        fontsize=8,
        color="#52514e",
    )
    ax.set(
        title=f"{cls} ({int(oof[cls].sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
axes[0].set_ylabel("precision")
fig.suptitle(
    "ResNet-18, out-of-fold rows of folds 0–4; dot = best-F1 threshold", color="#52514e"
)
fig.tight_layout()
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "full"}) as run:
    mlflow.log_table(summary.reset_index(names="fold"), "fold_scores.json")
    mlflow.log_table(noise.reset_index(names="metric"), "noise.json")
    flat = pooled.copy()
    flat.columns = [" ".join(c) for c in flat.columns]
    mlflow.log_table(flat.reset_index(names="class"), "pooled_f1.json")
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)